In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
sys.path.append("/home/nyxx/my_project/marej/")
import textgenutils as gutils
from src.models.gpt import GPT, GPT2BPETokenizer

ModuleNotFoundError: No module named 'nlp'

In [3]:
import transformers

In [4]:
import time

In [86]:
# model_hf = transformers.GPT2LMHeadModel.from_pretrained("gpt2").cuda()
# config = {
#     'n_layers': len(model_hf.transformer.h),
#     'n_heads': model_hf.transformer.h[0].attn.num_heads,
#     'embed_dim': model_hf.transformer.h[0].attn.embed_dim,
#     'vocab_size': model_hf.lm_head.out_features,
#     'block_size': model_hf.transformer.wpe.num_embeddings,
#     'dropout_p': model_hf.transformer.drop.p
# }

In [90]:
model = GPT.from_pretrained("gpt2").cuda()
# model_right = gpt_right.GPTRight.from_pretrained("gpt2").cuda()

In [91]:
def profile_kv_cache_generation_time_and_memory_consumption(prompt: str,
                                                            n_tokens_to_generate: int):
    tokenizer = GPT2BPETokenizer()
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model.decoder_blocks[0].attn.kv_cache[0].numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [92]:
profile_kv_cache_generation_time_and_memory_consumption(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 100
)

Loading file from cache: /home/nyxx/.cache/candle/gpt2_encoder.json
Loading file from cache: /home/nyxx/.cache/candle/gpt2_vocab.bpe
Prompt: Note: KV cache memory figures are only on single batch and a relatively small context length, and in practice will be much larger.
Tokens to generate: 100
Model Param Memory Consumption: 497.8 MB

NOT USING KV CACHE
Generation time:             0.7 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             0.5 sec
KV Cache Memory Consumption: 1.8 MB

Response: A large batch of memory can take a lot of time to run. For example, the KV cache size is quite large. If your main purpose is to cache and cache without requiring a lot of CPU overhead, the cache size should be in the order it is needed to support multiple processes on the same network.


The main benefit of using KV cache for caching is that the memory costs you to run and rebuild are much lower than for running KV as a separate thread, since K


In [24]:
def profile_kv_cache_generation_time_and_memory_consumption_right(prompt: str,
                                                            n_tokens_to_generate: int):
    tokenizer = GPT2BPETokenizer()
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model_right.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model_right, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model_right.decoder_blocks[0].attn._k_cache.numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [26]:
profile_kv_cache_generation_time_and_memory_consumption(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 100
)

Loading file from cache: /home/nyxx/.cache/candle/gpt2_encoder.json
Loading file from cache: /home/nyxx/.cache/candle/gpt2_vocab.bpe
Prompt: Note: KV cache memory figures are only on single batch and a relatively small context length, and in practice will be much larger.
Tokens to generate: 100
Model Param Memory Consumption: 497.8 MB

NOT USING KV CACHE
Generation time:             0.5 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             0.4 sec
KV Cache Memory Consumption: 1.8 MB

Response: sylv cutter 148 Sony Kaufman language languageDunptions cocoa enormDunDunDunDun perceptual UnchartedDun perceptual Unleashed Tony Unleashed Unleashed 1897 Tony pediatric Wise fermentation pediatric perceptualerence Himal insurobs editing 530 populist confusing intakefaLondon Unleashed Saur sol tragically tragically Tours669 torturedCloudutive censpolicy capsule Explan dagger enorm unsuccessful enorm Tobias enorm enorm░ reduced needs Hamilton clarify Hamilton Frozen 

In [30]:
profile_kv_cache_generation_time_and_memory_consumption_right(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 100
)

Loading file from cache: /home/nyxx/.cache/candle/gpt2_encoder.json
Loading file from cache: /home/nyxx/.cache/candle/gpt2_vocab.bpe
Prompt: Note: KV cache memory figures are only on single batch and a relatively small context length, and in practice will be much larger.
Tokens to generate: 100
Model Param Memory Consumption: 497.8 MB

NOT USING KV CACHE
Generation time:             0.6 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             0.5 sec
KV Cache Memory Consumption: 1.8 MB

Response: I thought this was a good idea because there is no guarantee that everything on the CPU's bus could read the data in the KV cache. It does provide some flexibility. For example, some CPUs only have two KV cache, and can read the data in KV cache.

KV caches are more efficient for memory than a memset. In most cases the KV cache is the first value (even if there's more than one value on it). In this case


In [ ]:
prompt = "hello world!"

In [ ]:
tokenizer = GPT2BPETokenizer()
indices = tokenizer.encode(prompt)
device = next(model.parameters()).device
indices = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

In [ ]:
out = model_hf(indices)
print(out["logits"].sum())

In [ ]:
out = model(indices)
print(out.sum())

In [ ]:
with torch.no_grad():
    for k in range(5):
        y = model_nano.generate(indices, 20, temperature=0.7, top_k=5)
        token = tokenizer.decode(y[0].tolist())
        indices_to_decode = []
        token = ''.join(token)
        print(token)
        print('---------------')

In [ ]:
import sys
sys.path.append("/home/nyxx/my_project/")
from nanoGPT.model import GPT2

In [ ]:
model_nano = GPT2.from_pretrained("gpt2").cuda()

In [ ]:
model.output_projection

In [ ]:
with torch.no_grad():
    for k in range(5):
        y = model.generate(indices, 20, temperature=0.7, top_k=5)
        token = tokenizer.decode(y[0].tolist())
        indices_to_decode = []
        token = ''.join(token)
        print(token)
        print('---------------')

In [ ]:
import torch

In [ ]:
causal_attn_mask = torch.triu(torch.ones(5, 5), diagonal=1)

In [ ]:
causal_attn_mask